<a href="https://colab.research.google.com/github/ttuhin/practice/blob/main/multi_agent_emergence_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulating Multi-Agent Environments with Emergent Behaviors

**Goal:** Build, experiment with, and analyze multi-agent simulations that produce emergent behaviors such as flocking, foraging, division of labor, and spatial segregation. The notebook contains runnable code (pure Python + PyTorch), visualization, evaluation metrics, and exercises.

**Outline**
1. Setup & reproducibility
2. Boids — rule-based flocking (emergence from simple rules)
3. Grid-world for resource foraging and predator-prey (rule-based + RL agents)
4. Multi-Agent RL (vanilla policy gradient) for cooperative/competitive tasks
5. Emergent communication: a toy discrete channel
6. Experiments & evaluation: metrics, ablations, visualization
7. Extensions & project ideas

Open this notebook in Google Colab or a local Jupyter environment. Some cells include optional pip installs for Colab (comment/uncomment as needed).


## 1 — Setup & Reproducibility
This cell installs optional packages (only for Colab), sets seeds, and imports required libraries.

In [1]:
# Install required packages for this notebook (into the active kernel)


# %pip install -q --upgrade pip


# %pip install -q torch torchvision matplotlib tqdm numpy

In [2]:
# Optional: uncomment in Colab if you want to install extra packages
# !pip install torch torchvision --quiet
# !pip install matplotlib tqdm --quiet

import math, random, time, os, json
from collections import deque, defaultdict
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Python, NumPy, PyTorch versions:')
import sys
print(sys.version.split('\n')[0])
print('numpy', np.__version__)
print('torch', torch.__version__)


Python, NumPy, PyTorch versions:
3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy 2.0.2
torch 2.8.0+cu126


## 2 — Boids: Rule-based flocking
Implement the classic Reynolds Boids with three simple rules: separation, alignment, and cohesion. Visualize and experiment with parameters to observe emergent flocking behavior.

In [ ]:
class Boids:
    def __init__(self, n=100, width=10.0, height=10.0, max_speed=0.35, max_force=0.05, perception=2.5):
        self.n = n
        self.w = width
        self.h = height
        self.max_speed = max_speed
        self.max_force = max_force
        self.perception = perception
        self.pos = np.random.rand(n,2) * np.array([width, height])
        angles = np.random.rand(n) * 2*np.pi
        self.vel = np.stack([np.cos(angles), np.sin(angles)], axis=1) * max_speed

    def _limit(self, v, max_norm):
        n = np.linalg.norm(v)
        if n > max_norm and n > 0:
            return v / n * max_norm
        return v

    def step(self, dt=0.2):
        new_vel = self.vel.copy()
        for i in range(self.n):
            p = self.pos[i]
            diffs = self.pos - p
            dists = np.linalg.norm(diffs, axis=1)
            nearby = (dists > 0) & (dists < self.perception)
            if nearby.sum() == 0:
                continue

            # Separation: steer away inversely with distance^2
            sep_vec = (-diffs[nearby] / (dists[nearby][:, None]**2 + 1e-6)).sum(axis=0)
            if np.linalg.norm(sep_vec) > 0:
                sep_vec = sep_vec / (np.linalg.norm(sep_vec) + 1e-9) * self.max_speed
            sep = self._limit(sep_vec - self.vel[i], self.max_force)

            # Alignment: match average heading
            avg_vel = self.vel[nearby].mean(axis=0)
            if np.linalg.norm(avg_vel) > 0:
                desired_align = avg_vel / np.linalg.norm(avg_vel) * self.max_speed
            else:
                desired_align = np.zeros(2)
            align = self._limit(desired_align - self.vel[i], self.max_force)

            # Cohesion: steer toward neighbors’ center of mass
            center = self.pos[nearby].mean(axis=0)
            to_center = center - p
            if np.linalg.norm(to_center) > 0:
                desired_coh = to_center / np.linalg.norm(to_center) * self.max_speed
            else:
                desired_coh = np.zeros(2)
            coh = self._limit(desired_coh - self.vel[i], self.max_force)

            # Weighted sum
            steer = 1.5*sep + 1.0*align + 1.0*coh
            steer = self._limit(steer, self.max_force)

            new_vel[i] = self._limit(self.vel[i] + steer, self.max_speed)

        self.vel = new_vel
        self.pos += self.vel * dt
        self.pos[:,0] %= self.w
        self.pos[:,1] %= self.h

# Quick demo plot/animation
boids = Boids(n=180, width=30, height=30, max_speed=0.35, max_force=0.05, perception=1)
fig, ax = plt.subplots(figsize=(8,8))
sc = ax.scatter(boids.pos[:,0], boids.pos[:,1], s=20)
ax.set_xlim(0, boids.w); ax.set_ylim(0, boids.h)
ax.set_aspect('equal', adjustable='box')
ax.set_title('Boids: flocking demo')

def update(frame):
    boids.step(dt=0.2)
    sc.set_offsets(boids.pos)
    return (sc,)

anim = animation.FuncAnimation(fig, update, frames=2000, interval=40, blit=False)
HTML(anim.to_jshtml())


## 3 — Grid-world: Foraging and Predator-Prey
A discrete grid world where agents forage (collect resources) and predators chase prey. We'll implement a simple environment and several agent policies (random, greedy, and heuristic).

In [ ]:
# Grid-world environment
class GridWorld:
    def __init__(self, size=(20,20), n_agents=5, n_predators=1, n_resources=50, vision=3, max_steps=200):
        self.h, self.w = size
        self.n_agents = n_agents
        self.n_predators = n_predators
        self.n_resources = n_resources
        self.vision = vision
        self.max_steps = max_steps
        self.reset()

    def reset(self):
        self.step_count = 0
        self.resources = set()
        while len(self.resources) < self.n_resources:
            self.resources.add((np.random.randint(self.h), np.random.randint(self.w)))
        self.agents = {}
        for i in range(self.n_agents):
            self.agents[i] = {'pos': (np.random.randint(self.h), np.random.randint(self.w)), 'alive': True, 'score':0}
        self.predators = {}
        for i in range(self.n_predators):
            self.predators[i] = {'pos': (np.random.randint(self.h), np.random.randint(self.w))}
        return self._get_obs()

    def _get_obs(self):
        obs = {}
        for i in range(self.n_agents):
            pos = self.agents[i]['pos']
            # local view: resources and predators within vision box
            view = []
            for dy in range(-self.vision, self.vision+1):
                row = []
                for dx in range(-self.vision, self.vision+1):
                    y = (pos[0]+dy) % self.h
                    x = (pos[1]+dx) % self.w
                    cell = {'resource': (y,x) in self.resources, 'predator': any(p['pos']==(y,x) for p in self.predators.values())}
                    row.append(cell)
                view.append(row)
            obs[i] = {'pos':pos, 'view':view}
        return obs

    def step(self, actions):
        # actions: dict agent_id -> one of 5 moves (stay, up, down, left, right)
        for aid, a in actions.items():
            if not self.agents[aid]['alive']:
                continue
            y,x = self.agents[aid]['pos']
            if a==1: y = (y-1)%self.h
            elif a==2: y = (y+1)%self.h
            elif a==3: x = (x-1)%self.w
            elif a==4: x = (x+1)%self.w
            self.agents[aid]['pos'] = (y,x)
            # collect resource
            if (y,x) in self.resources:
                self.resources.remove((y,x))
                self.agents[aid]['score'] += 1
        # predator moves greedily to nearest agent
        for pid, p in self.predators.items():
            py,px = p['pos']
            # find nearest alive agent
            best = None; bestd=1e9
            for aid,a in self.agents.items():
                if not a['alive']: continue
                ay,ax = a['pos']
                d = abs(py-ay)+abs(px-ax)
                if d<bestd:
                    bestd=d; best=(ay,ax)
            if best is None: continue
            ay,ax = best
            if py < ay: py+=1
            elif py>ay: py-=1
            if px < ax: px+=1
            elif px>ax: px-=1
            p['pos'] = (py%self.h, px%self.w)
            # check for capture
            for aid,a in self.agents.items():
                if a['pos']==p['pos']:
                    a['alive']=False
        self.step_count += 1
        done = self.step_count >= self.max_steps or len(self.resources)==0 or all(not a['alive'] for a in self.agents.values())
        return self._get_obs(), {i:self.agents[i]['score'] for i in self.agents}, done

# small demonstration run with random agents
env = GridWorld(size=(20,20), n_agents=6, n_predators=1, n_resources=60)
obs = env.reset()
for t in range(50):
    actions = {i: np.random.randint(0,5) for i in range(env.n_agents)}
    obs, scores, done = env.step(actions)
    if done:
        break
print('Scores after', t+1, 'steps:', scores)


## 4 — Multi-Agent Reinforcement Learning (vanilla policy gradient)
We implement a simple parameterized policy (small neural net) and train agents using REINFORCE in a cooperative foraging task. This is intentionally minimal to keep dependencies low; you can expand later to centralized critics or MADDPG.

In [ ]:
# Simple policy network and REINFORCE trainer (PyTorch)
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, hidden=64, n_actions=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )
    def forward(self, x):
        return nn.functional.softmax(self.net(x), dim=-1)

def obs_to_vec(agent_obs):
    # flatten local view into vector: resource and predator channels
    view = agent_obs['view']
    arr = np.array([[[1.0 if c['resource'] else 0.0, 1.0 if c['predator'] else 0.0] for c in row] for row in view])
    return arr.flatten().astype(np.float32)

# Create a cooperative foraging wrapper
class ForagingEnvWrapper:
    def __init__(self, grid_env):
        self.env = grid_env
        self.obs_dim = (2*((2*self.env.vision+1)**2))
        self.n_agents = self.env.n_agents

    def reset(self):
        obs = self.env.reset()
        return {i: obs_to_vec(obs[i]) for i in obs}

    def step(self, actions):
        obs, scores, done = self.env.step(actions)
        # reward = score increment (we compute delta per agent)
        rews = {i: self.env.agents[i]['score'] for i in self.env.agents}
        # return processed observations
        return {i: obs_to_vec(obs[i]) for i in obs}, rews, done

# REINFORCE training loop for independent agents
def train_reinforce(env_wrapper, n_epochs=200, episode_len=100, lr=1e-2):
    policies = {i: PolicyNet(env_wrapper.obs_dim) for i in range(env_wrapper.n_agents)}
    opts = {i: optim.Adam(policies[i].parameters(), lr=lr) for i in policies}
    gamma = 0.99
    for ep in range(n_epochs):
        obs = env_wrapper.reset()
        logps = {i: [] for i in policies}
        rewards = {i: [] for i in policies}
        for t in range(episode_len):
            actions = {}
            for i,pol in policies.items():
                x = torch.from_numpy(obs[i]).unsqueeze(0)
                probs = pol(x).squeeze(0).detach().numpy()
                a = np.random.choice(len(probs), p=probs)
                actions[i]=a
                logps[i].append(torch.log(pol(x).squeeze(0)[a] + 1e-9))
            obs, rews, done = env_wrapper.step(actions)
            for i in policies:
                rewards[i].append(rews[i])
            if done: break
        # compute returns and policy gradient update
        for i in policies:
            returns = []
            R = 0
            for r in reversed(rewards[i]):
                R = r + gamma*R
                returns.insert(0, R)
            returns = torch.tensor(returns)
            returns = (returns - returns.mean()) / (returns.std()+1e-8)
            loss = 0
            for lp, R in zip(logps[i], returns):
                loss += -lp * R
            opts[i].zero_grad()
            loss.backward()
            opts[i].step()
        if (ep+1)%20==0:
            total_scores = [env_wrapper.env.agents[i]['score'] for i in env_wrapper.env.agents]
            print(f'Epoch {ep+1}/{n_epochs}, scores per agent: {total_scores}')
    return policies

# Demo training (small)
gw = GridWorld(size=(12,12), n_agents=3, n_predators=0, n_resources=40, vision=2, max_steps=80)
fw = ForagingEnvWrapper(gw)
pols = train_reinforce(fw, n_epochs=80, episode_len=80, lr=3e-3)


## 5 — Emergent Communication (toy)
A cooperative task where one agent (sender) observes a resource location and sends a small discrete message to a receiver, which must navigate to the resource. Messages are discrete symbols; we'll learn policies using REINFORCE for both sender and receiver.

In [ ]:
# Toy emergent communication environment (continuous 2D)
class CommEnv:
    def __init__(self, n_symbols=4, max_steps=20):
        self.n_symbols = n_symbols
        self.max_steps = max_steps
        self.reset()

    def reset(self):
        # resource at random position in unit square
        self.resource = np.random.rand(2)
        self.receiver_pos = np.array([0.5, 0.5])
        self.step_count = 0
        return self.resource, self.receiver_pos.copy()

    def step(self, receiver_action):
        # receiver_action: 2D continuous movement vector clipped to small steps
        self.receiver_pos = np.clip(self.receiver_pos + np.clip(receiver_action, -0.1, 0.1), 0.0, 1.0)
        self.step_count += 1
        dist = np.linalg.norm(self.receiver_pos - self.resource)
        reward = -dist  # negative distance as reward (closer = better)
        done = self.step_count >= self.max_steps or dist < 0.05
        return (self.resource, self.receiver_pos.copy()), reward, done

# Simple sender & receiver policies (discrete sender, continuous receiver)
class Sender(nn.Module):
    def __init__(self, n_symbols=4):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2,32), nn.ReLU(), nn.Linear(32, n_symbols))
    def forward(self, x):
        logits = self.net(x)
        return torch.softmax(logits, dim=-1)

class Receiver(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 + 1, 64), nn.ReLU(), nn.Linear(64, 2))  # receives message id as scalar token
    def forward(self, x):
        return self.net(x)

def train_comm(env, n_epochs=500, lr=5e-3):
    sender = Sender(env.n_symbols)
    receiver = Receiver()
    opt = optim.Adam(list(sender.parameters()) + list(receiver.parameters()), lr=lr)
    for ep in range(n_epochs):
        resource, recv_pos = env.reset()
        # sender sends a symbol
        s_in = torch.from_numpy(resource.astype(np.float32)).unsqueeze(0)
        probs = sender(s_in).squeeze(0).detach().numpy()
        sym = np.random.choice(env.n_symbols, p=probs)
        # receiver acts for T steps conditioned on symbol
        total_reward = 0.0
        done = False
        state = (resource, recv_pos)
        for t in range(env.max_steps):
            # receiver input: current pos + symbol id normalized
            r_in = torch.from_numpy(np.concatenate([state[1], np.array([sym / max(1, env.n_symbols-1)])], dtype=np.float32)).unsqueeze(0)
            move = receiver(r_in).detach().numpy().squeeze(0)
            (res, pos), rew, done = env.step(move)
            total_reward += rew
            state = (res, pos)
            if done:
                break
        # REINFORCE-like update: we treat sender symbol as sampled action; receiver is deterministic here (we could add stochasticity)
        # For simplicity, compute surrogate loss: negative total_reward * log prob(symbol)
        s_probs = sender(s_in)
        logp = torch.log(s_probs[0, sym] + 1e-9)
        loss = - total_reward * logp
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (ep+1)%100==0:
            print(f'Epoch {ep+1}: avg total_reward {total_reward:.3f}')
    return sender, receiver

env = CommEnv(n_symbols=6, max_steps=20)
sender, receiver = train_comm(env, n_epochs=600, lr=1e-2)


## 6 — Experiments, Metrics & Ablations
Suggested experiments: vary agent density, resource density, predator strength, perception range, and reward shaping. Metrics to log: total reward, Gini coefficient of resource distribution, survival rate, spatial clustering (e.g., silhouette score on agent positions), and communication symbol entropy.

In [ ]:
# Utility: compute Gini coefficient for resource collection among agents
def gini(array):
    # array: list or numpy vector of non-negative values
    arr = np.array(array, dtype=np.float64)
    if arr.size == 0:
        return 0.0
    if np.all(arr==0):
        return 0.0
    arr = arr.flatten()
    arr_sorted = np.sort(arr)
    n = arr.size
    cum = np.cumsum(arr_sorted, dtype=float)
    g = (n+1 - 2*np.sum(cum)/cum[-1]) / n
    return g

# small demonstration:
gw = GridWorld(size=(16,16), n_agents=6, n_predators=1, n_resources=80)
fw = ForagingEnvWrapper(gw)
# run random baseline for comparison
scores_list=[]
for run in range(6):
    obs = fw.reset()
    for t in range(120):
        actions = {i: np.random.randint(0,5) for i in range(fw.n_agents)}
        obs, rews, done = fw.step(actions)
        if done: break
    scores = [gw.agents[i]['score'] for i in gw.agents]
    scores_list.append(scores)
    print('run', run+1, 'scores', scores, 'gini', gini(scores))
